## Matrix Multiplikation 3xN vs. Nx3
Author: Paul Christ

Date: 09.12.2025

Is there a diffrence in performance? 
Result: Yes, column major is approx 2 times faster.

In [11]:
import torch 
import numpy as np

In [12]:
torch.cuda.empty_cache()
#del q_DxN_new,q_NxD_new

In [ ]:
## Matrix dimensions
# M = N = K = 2**10

D = 3
n_particles = 2**27

dtype = torch.float32
device_gpu = 'cuda'


N_REPEATS_GPU = 2*12

''' Initialization of position matrices  '''
q_DxN = torch.randn(D, n_particles, device=device_gpu, dtype=dtype) 
q_NxD = torch.randn(n_particles, D, device=device_gpu, dtype=dtype) 

G_DxD = torch.randn(D, D, device=device_gpu, dtype=dtype)
q_DxN_new = torch.empty(D, n_particles, device=device_gpu, dtype=dtype)
q_NxD_new = torch.empty(n_particles, D, device=device_gpu, dtype=dtype)

# q_DxN = torch.randn(D, n_particles, device=device_gpu, dtype=dtype).contiguous()
# q_NxD = torch.randn(n_particles, D, device=device_gpu, dtype=dtype).contiguous()

# G_DxD = torch.randn(D, D, device=device_gpu, dtype=dtype).contiguous()

# q_DxN_new = torch.empty(D, n_particles, device=device_gpu, dtype=dtype).contiguous()
# q_NxD_new = torch.empty(n_particles, D, device=device_gpu, dtype=dtype).contiguous()




start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)


 

start_event.record()
for _ in range(N_REPEATS_GPU):
    q_DxN_new  = torch.matmul( G_DxD, q_DxN)
end_event.record()
torch.cuda.synchronize()
torch_time_DxN  = start_event.elapsed_time(end_event)  
 

start_event.record()
for _ in range(N_REPEATS_GPU):
    q_NxD_new  = torch.matmul( q_NxD, G_DxD)
end_event.record()
torch.cuda.synchronize()
torch_time_NxD  = start_event.elapsed_time(end_event)  
 

print("time for DxN:", torch_time_DxN)
print("time for NxD:", torch_time_NxD)
print(n_particles)

time for DxN: 359.88787841796875
time for NxD: 800.2713623046875
134217728


In [14]:
print(n_particles)

134217728


In [15]:
print("torch.cuda.memory_allocated: %fGB"%(torch.cuda.memory_allocated(0)/1024/1024/1024))
print("torch.cuda.memory_reserved: %fGB"%(torch.cuda.memory_reserved(0)/1024/1024/1024))
print("torch.cuda.max_memory_reserved: %fGB"%(torch.cuda.max_memory_reserved(0)/1024/1024/1024))

torch.cuda.memory_allocated: 6.007935GB
torch.cuda.memory_reserved: 7.521484GB
torch.cuda.max_memory_reserved: 7.521484GB
